In [42]:
import asyncio
import json
import time
from pprint import pprint

import requests
import websockets


# =============================================================================
# Configuration
# =============================================================================

N_MESSAGES = 4
CAPTURE_SECONDS = 20


MARKETS = {
    "spot": {
        "symbol": "BTC_USDT",
        "ws_url": "wss://api.gateio.ws/ws/v4/",
        "channel_prefix": "spot",
        "rest_book": "https://api.gateio.ws/api/v4/spot/order_book",
        "rest_params": {
            "currency_pair": "BTC_USDT",
            "limit": "100",
            "with_id": "true",
        },
    },

    "linear_perpetual": {
        "symbol": "BTC_USDT",
        "ws_url": "wss://fx-ws.gateio.ws/v4/ws/usdt",
        "channel_prefix": "futures",
        "rest_book": "https://api.gateio.ws/api/v4/futures/usdt/order_book",
        "rest_params": {
            "contract": "BTC_USDT",
            "limit": "100",
            "with_id": "true",
        },
    },

    "inverse_perpetual": {
        "symbol": "BTC_USD",
        "ws_url": "wss://fx-ws.gateio.ws/v4/ws/btc",
        "channel_prefix": "futures",
        "rest_book": "https://api.gateio.ws/api/v4/futures/btc/order_book",
        "rest_params": {
            "contract": "BTC_USD",
            "limit": "100",
            "with_id": "true",
        },
    },

    "linear_delivery": {
        "symbol": "BTC_USDT_20261009",
        "ws_url": "wss://fx-ws.gateio.ws/v4/ws/delivery/usdt",
        "channel_prefix": "futures",
        "rest_book": "https://api.gateio.ws/api/v4/delivery/usdt/order_book",
        "rest_params": {
            "contract": "BTC_USDT_20261009",
            "limit": "100",
            "with_id": "true",
        },
    },
}


# =============================================================================
# WebSocket collector
# =============================================================================

async def collect_gate_channel(
    market: str,
    config: dict,
    channel: str,
    payload: list,
    n: int = N_MESSAGES,
):
    messages = []

    async with websockets.connect(
        config["ws_url"],
        ping_interval=None,
        close_timeout=5,
    ) as ws:

        request = {
            "time": int(time.time()),
            "channel": channel,
            "event": "subscribe",
            "payload": payload,
        }

        await ws.send(
            json.dumps(request)
        )

        loop = asyncio.get_running_loop()
        deadline = loop.time() + CAPTURE_SECONDS

        while len(messages) < n:

            remaining = deadline - loop.time()

            if remaining <= 0:
                break

            try:
                raw = await asyncio.wait_for(
                    ws.recv(),
                    timeout=min(remaining, 5),
                )

            except asyncio.TimeoutError:
                continue

            if isinstance(raw, bytes):
                raw = raw.decode()

            try:
                message = json.loads(raw)

            except json.JSONDecodeError:
                print(
                    market,
                    channel,
                    "CONTROL:",
                    raw,
                )
                continue

            # Subscription acknowledgement.
            if message.get("event") == "subscribe":
                print(
                    market,
                    channel,
                    "SUBSCRIBE:",
                    message,
                )
                continue

            # Pong / other controls.
            if message.get("event") != "update":
                print(
                    market,
                    channel,
                    "CONTROL:",
                    message,
                )
                continue

            messages.append(message)

    return messages


# =============================================================================
# Market capture
# =============================================================================

async def inspect_gate_market(
    market: str,
    config: dict,
):
    symbol = config["symbol"]
    prefix = config["channel_prefix"]

    if market == "spot":

        trade_channel = "spot.trades"
        trade_payload = [symbol]

        book_channel = "spot.order_book_update"

        # pair, interval
        book_payload = [
            symbol,
            "100ms",
        ]

    else:

        trade_channel = f"{prefix}.trades"
        trade_payload = [symbol]

        book_channel = f"{prefix}.order_book_update"

        # contract, interval
        book_payload = [
            symbol,
            "100ms",
        ]

    trades, books = await asyncio.gather(

        collect_gate_channel(
            market,
            config,
            trade_channel,
            trade_payload,
        ),

        collect_gate_channel(
            market,
            config,
            book_channel,
            book_payload,
        ),
    )

    return {
        "trades": trades,
        "books": books,
    }


# =============================================================================
# Capture all
# =============================================================================

print("=" * 100)
print("SELECTED GATE.IO MARKETS")
print("=" * 100)

for name, config in MARKETS.items():
    print(
        f"{name:22}",
        config["symbol"],
    )


print()
print("=" * 100)
print("CAPTURING WEBSOCKET DATA")
print("=" * 100)


results = {}


for market, config in MARKETS.items():

    print()
    print("CAPTURING:", market)

    results[market] = await inspect_gate_market(
        market,
        config,
    )


# =============================================================================
# WebSocket report
# =============================================================================

for market, datasets in results.items():

    print()
    print("=" * 100)
    print(market.upper())
    print("=" * 100)

    # ---------------------------------------------------------------------
    # Trades
    # ---------------------------------------------------------------------

    trades = datasets["trades"]

    print()
    print("TRADES")
    print("-" * 100)

    print(
        "MESSAGES:",
        len(trades),
    )

    if trades:

        message = trades[0]

        print()
        print("ENVELOPE:")

        pprint({
            key: value
            for key, value in message.items()
            if key != "result"
        })

        result = message.get(
            "result"
        )

        print()
        print(
            "RESULT TYPE:",
            type(result).__name__,
        )

        if isinstance(result, dict):

            print(
                "TRADE FIELDS:"
            )

            print(
                sorted(result)
            )

            print()
            print(
                "FIRST TRADE:"
            )

            pprint(result)

        elif isinstance(result, list):

            print(
                "TRADES:",
                len(result),
            )

            if result:

                print(
                    "TRADE FIELDS:"
                )

                print(
                    sorted(result[0])
                )

                print()
                print(
                    "FIRST TRADE:"
                )

                pprint(
                    result[0]
                )

    # ---------------------------------------------------------------------
    # Books
    # ---------------------------------------------------------------------

    books = datasets["books"]

    print()
    print("BOOKS")
    print("-" * 100)

    print(
        "MESSAGES:",
        len(books),
    )

    previous_u = None

    for i, message in enumerate(
        books,
        start=1,
    ):

        result = message.get(
            "result",
            {},
        )

        print()
        print(
            f"MESSAGE {i}"
        )

        print("ENVELOPE:")

        pprint({
            key: value
            for key, value in message.items()
            if key != "result"
        })

        if not isinstance(
            result,
            dict,
        ):
            pprint(result)
            continue

        print()
        print("BOOK FIELDS:")

        print(
            sorted(result)
        )

        # Gate naming may differ by market.
        U = result.get("U")
        u = result.get("u")

        bids = (
            result.get("b")
            or result.get("bids")
            or []
        )

        asks = (
            result.get("a")
            or result.get("asks")
            or []
        )

        print()

        pprint({
            "s":
                result.get("s"),

            "t":
                result.get("t"),

            "U":
                U,

            "u":
                u,

            "previous_u":
                previous_u,

            "range_links_previous":
                (
                    U <= previous_u + 1 <= u
                    if (
                        previous_u is not None
                        and U is not None
                        and u is not None
                    )
                    else None
                ),

            "bids":
                len(bids),

            "asks":
                len(asks),
        })

        print()
        print("FIRST 3 BIDS:")

        pprint(
            bids[:3]
        )

        print()
        print("FIRST 3 ASKS:")

        pprint(
            asks[:3]
        )

        if bids:

            print()
            print(
                "BID LEVEL TYPE:",
                type(bids[0]).__name__,
            )

            print(
                "FIRST BID:"
            )

            pprint(
                bids[0]
            )

        if asks:

            print()
            print(
                "ASK LEVEL TYPE:",
                type(asks[0]).__name__,
            )

            print(
                "FIRST ASK:"
            )

            pprint(
                asks[0]
            )

        if u is not None:
            previous_u = u


# =============================================================================
# REST order books
# =============================================================================

print()
print("=" * 100)
print("REST ORDER BOOKS")
print("=" * 100)


rest_results = {}


for market, config in MARKETS.items():

    print()
    print("=" * 100)
    print(market.upper())
    print("=" * 100)

    response = requests.get(
        config["rest_book"],
        params=config["rest_params"],
        timeout=30,
    )

    print(
        "STATUS:",
        response.status_code,
    )

    print(
        "URL:",
        response.url,
    )

    if not response.ok:
        print(
            response.text[:1000]
        )
        continue

    payload = response.json()

    rest_results[market] = payload

    print()
    print(
        "TYPE:",
        type(payload).__name__,
    )

    if isinstance(
        payload,
        dict,
    ):

        print(
            "FIELDS:",
            sorted(payload),
        )

        pprint({
            "id":
                payload.get("id"),

            "current":
                payload.get("current"),

            "update":
                payload.get("update"),

            "bids":
                len(
                    payload.get(
                        "bids",
                        []
                    )
                ),

            "asks":
                len(
                    payload.get(
                        "asks",
                        []
                    )
                ),
        })

        print()
        print("FIRST 3 BIDS:")

        pprint(
            payload.get(
                "bids",
                []
            )[:3]
        )

        print()
        print("FIRST 3 ASKS:")

        pprint(
            payload.get(
                "asks",
                []
            )[:3]
        )


# =============================================================================
# Summary
# =============================================================================

print()
print("=" * 100)
print("SUMMARY")
print("=" * 100)


for market, config in MARKETS.items():

    print()

    pprint({
        "market":
            market,

        "instrument":
            config["symbol"],

        "trade_messages":
            len(
                results[
                    market
                ]["trades"]
            ),

        "book_messages":
            len(
                results[
                    market
                ]["books"]
            ),

        "rest_available":
            market in rest_results,
    })

SELECTED GATE.IO MARKETS
spot                   BTC_USDT
linear_perpetual       BTC_USDT
inverse_perpetual      BTC_USD
linear_delivery        BTC_USDT_20261009

CAPTURING WEBSOCKET DATA

CAPTURING: spot
spot spot.order_book_update SUBSCRIBE: {'time': 1790979879, 'time_ms': 1790979879445, 'conn_id': '18353ab7ba7bda19', 'trace_id': '7eeb7a3f74e60a7614d45836356836c3', 'channel': 'spot.order_book_update', 'event': 'subscribe', 'payload': ['BTC_USDT', '100ms'], 'result': {'status': 'success'}, 'requestId': '7eeb7a3f74e60a7614d45836356836c3'}
spot spot.trades SUBSCRIBE: {'time': 1790979879, 'time_ms': 1790979879445, 'conn_id': '7fffef69650771a3', 'trace_id': '74e9689a8f50e0920cbd982f08cb367f', 'channel': 'spot.trades', 'event': 'subscribe', 'payload': ['BTC_USDT'], 'result': {'status': 'success'}, 'requestId': '74e9689a8f50e0920cbd982f08cb367f'}

CAPTURING: linear_perpetual
linear_perpetual futures.trades SUBSCRIBE: {'time': 1790979885, 'time_ms': 1790979885853, 'conn_id': '702634b3659bb7e4